# Introduction.
## Technical prerequisites

You can run this demo code in Google Colab, which runs this notebook directly in the browser and does not require additional environment configuration. Watch [Introduction to Colab](https://www.youtube.com/watch?v=inN8seMm7UI), if you do not have a prior experience

Alternatively, to complete this quickstart locally, ensure that your development environment meets the following requirements:

-  Python 3.9+
-  An installation of [Jupyter](https://docs.jupyter.org/en/latest/install/notebook-classic.html) to run the notebook.

## Project overview

This notebook illustrates the simulation of a dialog between multiple parties. Each party is represented by an [agent](https://python.langchain.com/docs/modules/agents/), that uses a predefined large language model via [OpenAI API](https://openai.com/product) and [Langchain](https://python.langchain.com/docs/get_started/introduction) to participate in the conversation. Langchain tools are used to set up a communication framework between the agents.


`Part 1` guides a user for the specification of an agent using a dynamic custom-made dashboard

`Part 2` initiates a multiagent dialog and performs its analysis. The results are visualized in html format and saved in json file format.

**Important:** you need to run all cells to ensure a proper variable assingnment

# Part 1. Interacting with a user to define a custom-based specification for an agent.


Each code cell performs separate steps to run the project. Cells marked with (*) are optional and can be skipped.



In [ ]:
#@title Installing packages and dependencies

# %%capture is used to suppress the technical output during the package installation
%%capture

# Use specific versions of the packages to ensure the independency from the future releases
!pip install openai==1.7.0
!pip install typing_extensions==4.8.0
!pip install langchain==0.1.13

# installing potential searching tools
!pip install arxiv
!pip install duckduckgo-search
#!pip install googlesearch-python
!pip install wikipedia

# importing packages
from typing import List, Dict, Callable
from langchain.chains import ConversationChain
from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain.memory import ConversationBufferMemory
from langchain.prompts.prompt import PromptTemplate
from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    BaseMessage,
);

from langchain.agents import Tool
from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain.agents import load_tools

import os

## Optional usage of the Google Drive

If you run the code using Colab, your local variables (e.g. simulated dialogs) can be lost in case of an accidental disconnection. To ensure the storage of all materials you can link your working process to the google drive using an optional code cell below.

It creates the folder named *Multiagent_conversation* in your drive

In [ ]:
#@title Mounting the google drive and moving to a working directory (*)
from google.colab import drive
import os

drive.mount('/content/drive')
working_path = "/content/drive/MyDrive/Multiagent_conversation_26062024" # You can specify your own name of the directory

# Check if the folder exists
if os.path.isdir(working_path):
# If the folder exists, print a message
  print("The working folder already exists.")
else:
# If the folder does not exist, create it
  os.mkdir(working_path)
  print("The working folder has been created.")

# change to the specified path
%cd {working_path}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The working folder has been created.
/content/drive/MyDrive/Multiagent_conversation_26062024


## Setup your API key

Before you can use OpenAI API, you must first obtain an API key. If you don't already have one, you can check the follow up instructions in OpenAI website
<a class="button button-primary" href="https://openai.com/product" target="_blank" rel="noopener noreferrer">API key instructions</a>

If you have an API key, then you can import it either with

1.   Text file
2.   Adding the key to the secrets manager under the "🔑" in the left panel. Give it the name api_key and unable the access
3.   Directly specifying the variable in the workspace. Not preferred, since your API key will be directly visible

In [ ]:
#@title (1) Uploading API key with a text file.
import configparser
from google.colab import files
uploaded = files.upload()

# Iterate over the uploaded dictionary
for file_name in uploaded.keys():
# Join the current working directory with the file name
  api_file_path = os.path.join(os.getcwd(), file_name)
  print(f" API key file is uploaded to \n {api_file_path }")


# function to read the api key from the text file
def read_api_key(file_path):
    config = configparser.ConfigParser()
    config.read(file_path)
    api_key = config.get('API', 'key')
    return api_key

api_key = read_api_key(f'{api_file_path}')

Saving API_key.txt to API_key (1).txt
 API key file is uploaded to 
 /content/drive/MyDrive/Multiagent_conversation/API_key (1).txt


In [ ]:
#@title (2) Specification via the secret key
from google.colab import userdata
api_key=userdata.get('api_key')

In [ ]:
#@title (3) Direct variable specification
#api_key = 'your api key' #Place your api key here and uncomment the line

## Setup your dialog parties

At first we need to choose
the general topic of a conversation and

*   Broad topic of a simulated dialog
*   Number of parties (e.g. agents) and their names.
*(Optional) Custom prompt fields that will be further expanded at the second dashboard to provide the structured and detailed description of the agent. By default the fields are: `description`, `aim`, `conversation style`. `Description` field is mandatory, whereas other fields can be modified or deleted. Use buttons `Detele field` and `Add field` to delete or add new custom prompt fields

How to use the dashboard ?

1.   Fill in the corresponding dashboard fields.
2.   Click `Save` to save the inputs. Your inputs are saved in  `discussion.json` file stored in your working directory.
3.   Click `Clear the Input` to clear all fields. If you have previously saved your inputs in the json file, the file will not be removed
4.   Click `Delete saved input` to delete saved `discussion.json`


In [ ]:
#@title Dashboard for the initial agent specifications
import ipywidgets as widgets
from IPython.display import display
import json

# Save the current path as a storage path
storage_path = os.getcwd()
# Append 'discussion_setup.json' to the current path that will be used to store the conversation
setup_file_path  = os.path.join(storage_path, 'discussion_setup.json')


# Dashboard for the first part
topic_input = widgets.Text(description='Topic:')
parties_input = widgets.BoundedIntText(description='Parties:',value=0, min=0)
provide_names_checkbox = widgets.Checkbox(value=False, description='Provide names for the parties')
provide_fields_checkbox = widgets.Checkbox(value=False, description='Customize fields for the parties?')
add_field_button = widgets.Button(description='Add Field',button_style='primary', disabled=True)

names_vbox = widgets.VBox([])
fields_vbox = widgets.VBox([])

save_button1 = widgets.Button(description='Save', button_style='success', layout=widgets.Layout(width='100px'))
delete_button1 = widgets.Button(description='Clear the Input', button_style='danger', layout=widgets.Layout(width='120px'))

delete_file_button = widgets.Button(description= 'Delete saved input',layout=widgets.Layout(width='140px') )
delete_file_button.style.button_color = 'darkorange'

def update_names_input(change):
    if provide_names_checkbox.value:
        names_vbox.children = [widgets.Text(description=f'Person {i+1}:', value=f'Person {i+1}') for i in range(parties_input.value)]
    else:
        names_vbox.children = []

def create_delete_button():
    delete_button = widgets.Button(description='Delete field', layout=widgets.Layout(width='100px'))
    delete_button.style.button_color = 'salmon'
    return delete_button

def update_fields_input(change):
    if provide_fields_checkbox.value:
        add_field_button.disabled = False
        fields_vbox.children = [
            widgets.HBox([widgets.Text(value='Description', disabled=True)]),
            widgets.HBox([widgets.Text(value='Aim'),create_delete_button()]),
            widgets.HBox([widgets.Text(value='Conversation Style'),create_delete_button()])
                                ]
        for hbox in fields_vbox.children[1:]:
            hbox.children[1].on_click(delete_field)

    else:
        add_field_button.disabled = True
        fields_vbox.children = []

def delete_field(b):
    # Create a new list for children without the deleted field, but always keep the Description field
    new_children = [fields_vbox.children[0]]  # Start with the Description field
    for child in fields_vbox.children[1:]:  # Skip the Description field in this loop
        if child.children[1] != b:  # If the delete button in this child was not the one clicked, keep the field
            new_children.append(child)
    fields_vbox.children = new_children  # Assign the new list, which still includes the Description field


# Update add_field function to include a delete button
def add_field(b):
    field_number = len(fields_vbox.children) + 1
    new_field = widgets.HBox([
        widgets.Text(),
        create_delete_button()
    ])
    new_field.children[1].on_click(delete_field)
    fields_vbox.children = list(fields_vbox.children) + [new_field]

def save_to_json1(b):
    data = {
        'Topic': topic_input.value,
        'Number of Interacting People': parties_input.value,
        'Names': [name_input.value for name_input in names_vbox.children] if provide_names_checkbox.value else [f'Person {i+1}' for i in range(parties_input.value)],
        'Custom Fields': [field_input.children[0].value for field_input in fields_vbox.children] if provide_fields_checkbox.value else ['Description', 'Aim', 'Style']
    }
    with open(setup_file_path, 'w') as f:
        json.dump(data, f)
        print(f"File has been created in {setup_file_path}")


def delete_inputs1(b):
    topic_input.value = ''
    parties_input.value = 0
    names_vbox.children = []
    fields_vbox.children = []
    provide_names_checkbox.value= False
    provide_fields_checkbox.value= False
    print(f"All inputs have been cleared")

# Function to check and delete the file
def check_and_delete(file_path):
    if os.path.isfile(file_path):
      os.remove(file_path)
      print(f"File has been deleted from {setup_file_path}")
    else:
      print("File does not exist.")

def delete_on_button_click(b):
    check_and_delete(setup_file_path)

provide_names_checkbox.observe(update_names_input, names=['value'])
parties_input.observe(update_names_input, names=['value'])
provide_fields_checkbox.observe(update_fields_input, names=['value'])

add_field_button.on_click(add_field)
save_button1.on_click(save_to_json1)
delete_button1.on_click(delete_inputs1)
delete_file_button.on_click(delete_on_button_click)

buttons_hbox = widgets.HBox([save_button1, delete_button1,delete_file_button])
fields_control_hbox = widgets.HBox([provide_fields_checkbox, add_field_button])

display(topic_input, parties_input, provide_names_checkbox, names_vbox, fields_control_hbox, fields_vbox, buttons_hbox)


Text(value='', description='Topic:')

BoundedIntText(value=0, description='Parties:')

Checkbox(value=False, description='Provide names for the parties')

VBox()

VBox()

File has been created in /content/drive/MyDrive/Multiagent_conversation_26062024/discussion_setup.json


In [ ]:
#@title Check what inputs are now stored in the `discussion_setup.json`
import os
import json

with open(setup_file_path, 'r') as f:
  data = json.load(f)
  print(data)

{'Topic': 'Should hunting of wild boars be forbidden in Germany?', 'Number of Interacting People': 3, 'Names': ['Hunter', 'Farmer', 'Animal protection representative'], 'Custom Fields': ['Description', 'Short-term goals', 'General aim', 'Scientific foundation', 'Ethical foundation']}


In [ ]:
#@title Example of getting a specific part (e.g. Topic) of stored `discussion_setup.json` file
def get_discussion_topic(json_file_path):
    try:
        with open(json_file_path, 'r') as file:
            data = json.load(file)
            return data.get('Topic', 'Unknown Topic')  # Default to 'Unknown Topic' if not found
    except FileNotFoundError:
        print("JSON file not found.")
        return 'Unknown Topic'

discussion_topic = get_discussion_topic(setup_file_path)
print(discussion_topic)

Should hunting of wild boars be forbidden in Germany?


## Setup detailed specifications of dialog parties

Once `topic`, `parties` and `prompt fields`are specified we need to provide more details in `prompt fields` assigned to each party to further elaborate the parties descriptions. This can be done using another custom-adjusted dashboard

How to use the dashboard ?

1.   Fill in the corresponding dashboard fields (manually or automatically). See details below.
2.   Click `Save` to save the inputs. Your inputs are saved in  `discussion_details.json` file stored in your working directory.
3.   Click `Clear the Input` to clear all fields. If you have previously saved your inputs in the json file, the file will not be removed
4.   Click `Delete saved input` to delete saved `discussion_details.json`

The prompt field descriptions can be provided manually or automatically by selecting option `Automatic` at the bottom of a party column. In `Automatic` scenario, you will be asked to define parameters of a model to be used for the OpenAI API call to fill the prompt fields assigned to the party. You can choose what fields should be filled with GPT and what fields should remain untouched. The `name` of the party as well as the input from `Description` field are used to fill the prompt fields.

The following parameters can be adjusted for `Automatic` filling.

$ \underline{\text{Model}} $: By default it is `gpt-4`, but, if you do not have this model available for you, then you can switch to default `gpt-3.5-turbo`. Alternatively, you can specify more elaborate text generation models, such as `davinci-002`. You can check the list of models with their rate limits in [OpenAI Platform](https://platform.openai.com/). Log in and go to *Settings-Organization-Limits*.

$ \underline{\text{Temperature}} $: Affects the variability of the model output. It is assumed to modify the un-normalized log-probabilities (e.g. logits) and bring them closer to the uniform distribution. You can check the following pages from [Microsoft](https://learn.microsoft.com/en-us/answers/questions/1313865/recommended-openai-temperature-and-top-p) and user cases from [Medium](https://aviralrma.medium.com/understanding-llm-parameters-c2db4b07f0ee) for more information.

Lower values for temperature result in more consistent outputs (e.g. 0.2), while higher values generate more diverse and creative results (e.g. 1.0). Select a temperature value based on the desired trade-off between coherence and creativity for your specific application. The temperature can range is from 0 to 2.

$ \underline{\text{Frequency penalty}} $: Used to reduce the repetition of the same words. It directly modifies logits with an additive contribution. You can read more about it in [OpenAI parameter details page](https://platform.openai.com/docs/guides/text-generation/parameter-details)

Reasonable values for the penalty coefficients are around 0.1 to 1 if the aim is to just reduce repetitive samples somewhat. If the aim is to strongly suppress repetition, then one can increase the coefficients up to 2, but this can noticeably degrade the quality of samples. Negative values can be used to increase the likelihood of repetition.

$ \underline{\text{Output word limit}} $: Sets the output words limit




In [ ]:
#@title Dashboard for detailed agent specifications
import ipywidgets as widgets
from IPython.display import display
import json
import openai

import os
from openai import OpenAI

#### Load the discussion settings from discussion_setup.json ####

# General function to get the information from json file
def get_discussion_topic(json_file_path):
    try:
        with open(json_file_path, 'r') as file:
            data = json.load(file)
            return data.get('Topic', 'Unknown Topic')  # Default to 'Unknown Topic' if not found
    except FileNotFoundError:
        print("JSON file not found.")
        return 'Unknown Topic'

# Load the names and fields for parties and create widgets from a specific json file
def load_parties_and_fields():
    try:
        with open(setup_file_path, 'r') as file: #uploading json file with saved setup instructions
            data = json.load(file)
            party_names = data.get('Names')
            custom_fields = data.get('Custom Fields')
    except FileNotFoundError:
        print ('Setup file was not found')
    return create_party_widgets(party_names, custom_fields)


discussion_topic = get_discussion_topic(setup_file_path) # setup_file_path string variable contains setup configurations
detailed_file_path=os.path.join(storage_path,'discussion_details.json') # file that stores the discussion details, storage_path variable was defined within a previous dashboard.


#### Specifying additional functions later used in the dashboard ####

# Function to call OpenAI for an automatic generation of agent descriptions
def generate_agent_description(name, description, field_label, gpt_model_name, gpt_temperature, word_limit, freq_penalty):
    agent_descriptor_system_message = SystemMessage(content=f"You are supposed to simulate the behaviour of a person with a specific description.")
    agent_specifier_prompt = [agent_descriptor_system_message,
        HumanMessage(
            content=f"Please generate a creative potential {field_label} for a {name} with the following {description} for the  \
            discussion on {discussion_topic} to persuade others in his/her point of view. Make your response to {word_limit} words or less"
        ),
    ]
    agent_description = ChatOpenAI(temperature=gpt_temperature, model_name=gpt_model_name, openai_api_key=api_key, model_kwargs={"frequency_penalty": freq_penalty})(agent_specifier_prompt).content
    return agent_description

# Function to implement an option for an automatic filling of the agent description with GPT
def create_gpt_fill_button_handler(party_name, description_text_area, other_text_areas, field_labels,
                                   model_name_widget, temperature_widget, word_limit_widget, freq_penalty_widget, checkboxes):
    def on_gpt_fill_button_clicked(b):
        # Iterate through the pairs of text area and labels
        for text_area, label in zip([description_text_area] + other_text_areas, ['Description'] + field_labels):
            if checkboxes[label].value: # checking if the corresponding field should be automatically filled
              generated_text = generate_agent_description(party_name, description_text_area.value, label,
                                                          model_name_widget.value,
                                                          temperature_widget.value,
                                                          word_limit_widget.value,
                                                          freq_penalty_widget.value,
                                                          )

              text_area.value = generated_text  # Put generated text in the corresponding text area

    return on_gpt_fill_button_clicked

# Function to display additional widgets, if 'Automatic' filling option is selected
def create_automatic_change_handler(description_text_area, other_text_areas, field_labels, auto_widgets, fill_button_container, party_name,custom_fields):
    def on_automatic_change(change):
        if change['new']:  # If the checkbox is checked

            # Creating widgets to specify model name and temperature for the automatic filling
            model_name_widget = widgets.Text(description='Model:',value='gpt-4')
            temperature_widget = widgets.FloatSlider(description='Temperature:',style={'description_width': '150px'},value=0.7, min=0.0, max=2.0, step=0.1)
            freq_penalty_widget = widgets.FloatSlider(description='Frequency penalty:',style={'description_width': '150px'}, value=0, min=0.0, max=2.0, step=0.1,)
            word_limit_widget = widgets.IntText(description='Output word limit', value=30, style={'description_width': '150px'})
            checkboxes_widget= create_vertical_checkboxes(custom_fields)

            # Extract checkbox widgets from the vbox (Assuming ordering and label consistency)
            checkboxes_dict = {field: checkboxes_widget.children[i] for i, field in enumerate(['All'] + custom_fields)}


            fill_button = widgets.Button(description='Fill with GPT', button_style='primary', layout=widgets.Layout(width='auto'))

            fill_button.on_click(create_gpt_fill_button_handler(party_name, description_text_area, other_text_areas, field_labels,
                                                                model_name_widget, temperature_widget, word_limit_widget, freq_penalty_widget, checkboxes_dict))

            auto_widgets.children = [model_name_widget, temperature_widget, freq_penalty_widget, word_limit_widget, checkboxes_widget]
            fill_button_container.children = [fill_button]

        else: # If the checkbox is unchecked then widgets are removed
            auto_widgets.children = []
            fill_button_container.children = []

    return on_automatic_change

# Function that creates descriptions for the Widgets, used in create_party_widgets
def create_description_label(description):
     return widgets.Label(value=description)

# Function to  to create checkboxes to allow the user to control the automatic filling
def create_vertical_checkboxes(fields):
    # Initialize the dictionary to store the checkbox widgets
    checkboxes = {}

    # Create the 'All' checkbox and set it to be checked by default
    all_checkbox = widgets.Checkbox(value=True, description='All', indent=False)
    checkboxes['All'] = all_checkbox

    # Create the other checkboxes based on the custom fields, all checked and disabled by default
    for field in fields:
        checkbox = widgets.Checkbox(value=True, description=field, disabled=True, indent=False)
        checkboxes[field] = checkbox

    # Define the callback function for changes in the 'All' checkbox
    def on_all_change(change):
        if change['new']:  # If 'All' is checked
            for key in checkboxes:
                if key != 'All':  # Skip the 'All' checkbox itself
                    checkboxes[key].value = True  # Check the checkbox
                    checkboxes[key].disabled = True  # Disable the checkbox
        else:  # If 'All' is unchecked
            for key in checkboxes:
                if key != 'All':  # Skip the 'All' checkbox itself
                    checkboxes[key].disabled = False  # Enable the checkbox

    # Attach the callback to the 'All' checkbox
    all_checkbox.observe(on_all_change, names='value')

    # Create a VBox to arrange the checkboxes vertically
    vbox = widgets.VBox(list(checkboxes.values()))

    return vbox

#### Create widgets for the individual negotiation parties. Widgets are organized as vertical columns. One column per a party ####

def create_party_widgets(party_names, custom_fields):
    columns = []
    for i, name in enumerate(party_names):
        field_widgets = []
        description_text_area = []
        other_text_areas = []
        field_labels = []

        for field in custom_fields:
            text_area = widgets.Textarea(layout=widgets.Layout(width='95%', height='150px'))
            field_widgets.append(widgets.VBox([create_description_label(field), text_area]))

            if field == 'Description':
                description_text_area = text_area
            else:
                other_text_areas.append(text_area)
                field_labels.append(field)

        auto_widgets = widgets.VBox([])
        fill_button_container = widgets.VBox([])

        automatic_checkbox = widgets.Checkbox(value=False, description='Automatic')
        automatic_change_handler = create_automatic_change_handler(description_text_area, other_text_areas,
                                                                   field_labels, auto_widgets, fill_button_container,name, custom_fields)
        automatic_checkbox.observe(automatic_change_handler, names='value')

        party_label = widgets.Label(value=name)
        vbox = widgets.VBox([party_label] + field_widgets + [automatic_checkbox, auto_widgets, fill_button_container], layout=widgets.Layout(margin='0px 5px 0px 5px'))
        columns.append(vbox)

    hbox = widgets.HBox(columns, layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))
    return hbox



# Function to save data to a separate JSON named 'discussion_details.json'
def save_to_json2(b):
    all_party_info = []  # Use this list to collect information from all parties.
    for vbox in party_widgets.children:
        party_info = {}  # Dictionary to store the current party's information.
        # The first child of the vbox is the party name label, we assume this structure is constant.
        party_info['Name'] = vbox.children[0].value  # Directly assign the party name value.

        # Iterate over the remaining children in the vbox which are supposed to be field containers.
        for field_vbox in vbox.children[1:]:  # Skip the first child, which is the party name.
            if isinstance(field_vbox, widgets.VBox) and len(field_vbox.children) > 1:
                # Make sure the first child of this container is indeed a label before accessing its value.
                if isinstance(field_vbox.children[0], widgets.Label):
                    field_label = field_vbox.children[0].value.rstrip(':')  # Retrieve the field label.
                    field_value = field_vbox.children[1].value  # Retrieve the field value.
                    party_info[field_label] = field_value  # Add the field info to the party's data dictionary.

        all_party_info.append(party_info)  # Add the party's info to the overall list.

    # Write all collected party information to the JSON file.
    if all_party_info:  # Check if there's anything to save.
        with open(detailed_file_path, 'w') as f:
            json.dump(all_party_info, f)  # Save the JSON data.
            print(f"File has been created in {detailed_file_path}.")


# Function to delete all generated inputs for the dashboard and to restore default settings of 'Automatic' filling
def delete_inputs2(b):
    # Iterate over each party's VBox
    for party_vbox in party_widgets.children:
        # Iterate over the field VBoxes inside the party VBox, starting from the second child (index 1)
        for field_vbox in party_vbox.children[1:]:  # Skipping the party name label at index 0
            if isinstance(field_vbox, widgets.VBox) and len(field_vbox.children) >= 2:
                # Now iterate over each widget in the VBox to handle different widget types
                for widget in field_vbox.children:
                    # Check if the widget is another VBox (possibly containing checkboxes)
                    if isinstance(widget, widgets.VBox):
                        # If so, iterate through its children to find checkboxes
                        for sub_widget in widget.children:
                            if isinstance(sub_widget, widgets.Checkbox):
                                # Reset all checkboxes, specifically looking for 'All'
                                sub_widget.value = True  # Reset all checkboxes to True
                    # Reset other types of widgets
                    elif isinstance(widget, widgets.Textarea):
                        widget.value = ''
                    elif isinstance(widget, widgets.Text):
                        widget.value = 'gpt-4'  # Reset text widgets
                    elif isinstance(widget, widgets.FloatSlider):
                        # Check the description to determine which slider it is and reset accordingly
                        if widget.description == 'Temperature:':
                            widget.value = 0.7  # Reset temperature slider
                        elif widget.description == 'Frequency penalty:':
                            widget.value = 0  # Reset frequency penalty slider
                    elif isinstance(widget, widgets.IntText):
                        widget.value = 30  # Reset integer text widgets



# Function to check and delete the stored 'discussion_details.json' file
def check_and_delete(file_path):
    if os.path.isfile(file_path):
      os.remove(file_path)
      print(f"File has been deleted from {file_path}.")
    else:
      print("File does not exist.")

def delete_on_button_click(b):
    check_and_delete(detailed_file_path)

#### Bringing all widgets to the dashboard ####

party_widgets = load_parties_and_fields()

save_button2 = widgets.Button(description='Save', button_style='success', layout=widgets.Layout(width='100px'))
save_button2.on_click(save_to_json2)

delete_button2 = widgets.Button(description='Clear the Input', button_style='danger', layout=widgets.Layout(width='120px'))
delete_button2.on_click(delete_inputs2)

delete_file_button = widgets.Button(description= 'Delete saved input',layout=widgets.Layout(width='140px') )
delete_file_button.style.button_color = 'darkorange'
delete_file_button.on_click(delete_on_button_click)

h_save_delete_2=widgets.HBox([save_button2,delete_button2,delete_file_button ], layout=widgets.Layout(flex_flow='row wrap'))

# Display widgets
display(party_widgets, h_save_delete_2)

File has been created in /content/drive/MyDrive/Multiagent_conversation_26062024/discussion_details.json.


# Part 2. Simulating the conversation based on the custom-defined data (saved in json format)

 ## Outlining the dialog playground and preparing agents based on provided information

 Below we specify a dialogue simulation system tailored to handle interactions between multiple agents in a conversation. Each agent belog to a class `DialogueAgent` and `DialogSimulator` classs handles the conversation flow between the agents.

 Then we define a subclass of `DialogueAgent` named `DialogueAgentWithTools`. This subclass enhances the functionality of the original `DialogueAgent` by equipping it with additional tools for processing or responding within dialogues.

In [ ]:
#@title Outlining Agents, Dialog simulator and equipping agents with tools
class DialogueAgent:
    def __init__(self, name: str, system_message: SystemMessage,model: ChatOpenAI):
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self):
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model([self.system_message,HumanMessage(content="\n".join(self.message_history + [self.prefix]))])
        return message.content

    def receive(self, name: str, message: str):
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")


class DialogueSimulator: #initiating dialog between the agents
    def __init__(self,agents: List[DialogueAgent],selection_function: Callable[[int, List[DialogueAgent]], int]):
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message

class DialogueAgentWithTools(DialogueAgent): # equipping agents with tools
    def __init__(self,name: str,system_message: SystemMessage,model: ChatOpenAI,tool_names: List[str],**tool_kwargs):
        super().__init__(name, system_message, model)
        self.tools = load_tools(tool_names, **tool_kwargs)

    def send(self):
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        agent_chain = initialize_agent(self.tools,self.model,
            agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
            verbose=True,
            memory=ConversationBufferMemory(
                memory_key="chat_history", return_messages=True
            ),
        )
        message = AIMessage(
            content=agent_chain.run(
                input="\n".join(
                    [self.system_message.content] + self.message_history + [self.prefix]
                )
            )
        )

        return message.content

## Assigning tools to the agents

At this section we assign tools to each agent.

In [ ]:
#@title Uploading the custom information from the stored json file and assign resources
import json
def extract_topic_from_json(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
        topic = data.get('Topic')
        names = data.get('Names')
    return topic,names

topic, names_list= extract_topic_from_json(setup_file_path)

tool_list=["arxiv", "ddg-search", "wikipedia"] # already predifined tools from LangChain
names = {name: tool_list for name in names_list}

print(topic, names)


Should hunting of wild boars be forbidden in Germany? {'Hunter': ['arxiv', 'ddg-search', 'wikipedia'], 'Farmer': ['arxiv', 'ddg-search', 'wikipedia'], 'Animal protection representative': ['arxiv', 'ddg-search', 'wikipedia']}


In [ ]:
setup_file_path
detailed_file_path


'/content/drive/MyDrive/Multiagent_conversation_26062024/discussion_details.json'

In [ ]:
#@title Retrieve a prompt for each agent
def structure_agent_descriptions(json_file_path):
    """
    Opens a JSON file containing lists with information about persons.
    Structures the stored information into a dictionary in the format {name: agent_description}.
    """
    agent_descriptions = {}

    try:
        with open(json_file_path, 'r') as file:
            data = json.load(file)
            # Ensure that data is a list
            if not isinstance(data, list):
                return "Invalid file format: Expected a list."

            # Iterate through each list (person's info) in the data
            for person_info in data:
                # Ensure each person's info is in dictionary format
                if not isinstance(person_info, dict):
                    continue  # Skip if the format is not a dictionary

                # Retrieve the name and construct the description
                name = person_info.get("Name", "Unknown")
                agent_description = ". ".join([f"{key}: {value}" for key, value in person_info.items()if key != "Name"])
                agent_descriptions[name] = agent_description

            return agent_descriptions
    except FileNotFoundError:
        return "File not found: " + json_file_path
    except json.JSONDecodeError as e:
        return "Error decoding JSON: " + str(e)
    except Exception as e:
        return "An error occurred: " + str(e)



In [ ]:
agent_descriptions = structure_agent_descriptions(detailed_file_path)
agent_descriptions

{'Hunter': 'Description: As an expert in animal ecology, especially of the wild boar, conservationist, but also ethical hunter, I do believe that a proper management of the wild boar population, including sustainable hunting, is crucial for Germany, for several important reasons. Wild boar (Sus scrofa ferus) is one of the most widespread large mammal species in the world, characterized by high plasticity and ability to adapt to new environments and environmental changes. Despite challenges such as disease, their population in Europe is growing exponentially. Based on current knowledge, recreational hunting is the reasonable method to counteract the overpopulation of wild boar, which I will explain further.\nMaintaining both the population size and a balanced age structure is essential to prevent the spread of disease, reduce damage to crops and forests, and mitigate human-wildlife conflicts. In addition, sustainable hunting practices contribute to the overall health of ecosystems and p

In [ ]:
storage_path

'/content/drive/MyDrive/Multiagent_conversation_26062024'

In [ ]:
file_name = 'agent_descriptions.json'

# Ensure the storage path exists
os.makedirs(storage_path, exist_ok=True)

# Construct the full file path
file_path = os.path.join(storage_path, file_name)

# Save the dictionary to a JSON file
with open(file_path, 'w') as json_file:
    json.dump(agent_descriptions, json_file, indent=4)

print(f"Dictionary saved to {file_path}")

Dictionary saved to /content/drive/MyDrive/Multiagent_conversation_26062024/agent_descriptions.json


In [ ]:
#@title Generate a guiding message for each agent
def generate_system_message(name, description, tools):
    return f"""

Your name is {name}.

Your description is as follows: {description}

Your goal is to persuade your conversation partner of your point of view.

DO look up information with your tool to refute your partner's claims.
DO cite your sources.

DO NOT fabricate fake citations.
DO NOT cite any source that you did not look up.

Do not add anything else.

Stop speaking the moment you finish speaking from your perspective.
"""


agent_system_messages = {
    name: generate_system_message(name, description, tools)
    for (name, tools), description in zip(names.items(), agent_descriptions.values())
}

In [ ]:
for name, system_message in agent_system_messages.items():
    print(name)
    print(system_message)

Hunter


Your name is Hunter.

Your description is as follows: Description: As an expert in animal ecology, especially of the wild boar, conservationist, but also ethical hunter, I do believe that a proper management of the wild boar population, including sustainable hunting, is crucial for Germany, for several important reasons. Wild boar (Sus scrofa ferus) is one of the most widespread large mammal species in the world, characterized by high plasticity and ability to adapt to new environments and environmental changes. Despite challenges such as disease, their population in Europe is growing exponentially. Based on current knowledge, recreational hunting is the reasonable method to counteract the overpopulation of wild boar, which I will explain further.
Maintaining both the population size and a balanced age structure is essential to prevent the spread of disease, reduce damage to crops and forests, and mitigate human-wildlife conflicts. In addition, sustainable hunting practices co

In [ ]:
#@title Bring a moderator to manage a talk
#word_limit=50
topic_specifier_prompt = [
    SystemMessage(content="You can make a topic more specific."),
    HumanMessage(
        content=f"""{topic}

        You are the moderator.
        Please organise the discussion to reach the following goals

        The participants need to formulate a clear and precise formulation of the topic related issues

        The issue formulation should take into the account the following:
          The issues should cover different aspects of the problem
          The issue definition should be agreed between stakeholders

          Not only provide your suggestions, but, also, comment on the issues  raised by other parties.
          Implement the feedback from the other parties into your response.

        Speak directly to the participants: {*names,}.
        Do not add anything else."""
    ),
]
specified_topic = ChatOpenAI(temperature=1, openai_api_key = api_key)(topic_specifier_prompt).content

#print(f"Original topic:\n{topic}\n")
#print(f"Detailed topic:\n{specified_topic}\n")

In [ ]:
#@title Elaborated topic by GPT-4
import ipywidgets as widgets
from IPython.display import display

# Assuming specified_topic and topic are defined somewhere

# Create labels as widget titles
specified_topic_label = widgets.Label('Specified Topic:')
original_topic_label = widgets.Label('Original Topic:')


# Create text area widget for specified_topic without the description attribute
specified_topic_text = widgets.Textarea(
    value=specified_topic,
    disabled=False,
    layout=widgets.Layout(width='400px', height='300px')
)

# Create text area widget for original_topic, but make it read-only, without the description attribute
original_topic_text = widgets.Textarea(
    value=topic,
    disabled=True,  # This makes the field read-only
    layout=widgets.Layout(width='400px', height='50px')
)

# Create a Save button
save_button = widgets.Button(
    description='Save',
    button_style='success',  # Makes the button green
    tooltip='Click to save the specified topic'
)

# Function to update specified_topic variable with the new content
def save_new_topic(b):
    global specified_topic
    specified_topic = specified_topic_text.value
    print("New specified topic saved")



# Button click event
save_button.on_click(save_new_topic)

# Grouping labels with their respective text areas using VBox for vertical alignment
specified_topic_group = widgets.VBox([specified_topic_label, specified_topic_text])
original_topic_group = widgets.VBox([original_topic_label, original_topic_text])

# Display widgets together using HBox for horizontal alignment
display(widgets.HBox([specified_topic_group, original_topic_group]), save_button)


Button(button_style='success', description='Save', style=ButtonStyle(), tooltip='Click to save the specified t…

New specified topic saved
New specified topic saved


In [ ]:
specified_topic

'Discuss the following topic: \n\n"To what extent should hunting of wild boars be allowed in Germany particularly considering population control, socio-economical impacts, and ethical considerations?" \n\nEach of you is a specialist in the area and your task is to effectively communicate your position to the other parties\n\nPlease, also, comment on the suggestions raised by other parties. Explain your position.\n\n\n'

In [ ]:
#@title Specifying agents
# we set `top_k_results`=2 as part of the `tool_kwargs` to prevent results from overflowing the context limit
agents = [
    DialogueAgentWithTools(
        name=name,
        system_message=SystemMessage(content=system_message),
        model=ChatOpenAI(model_name="gpt-4", temperature=0.7, openai_api_key = api_key),
        tool_names=tools,
        top_k_results=2,
    )
    for (name, tools), system_message in zip(
        names.items(), agent_system_messages.values()
    )
]

In [ ]:
#@title Specifying the mechanism to choosing the next speaker
def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    idx = (step) % len(agents)
    return idx

In [ ]:
#@title Simulating the conversation
max_iters = 12
n = 0

# Initialize a list to hold the conversation
conversation = []

simulator = DialogueSimulator(agents=agents, selection_function=select_next_speaker)
simulator.reset()
simulator.inject("Moderator", specified_topic)
print(f"(Moderator): {specified_topic}")
print("\n")

conversation.append({"name": "Moderator", "message": specified_topic})

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}")
    print("\n")

    conversation.append({"name": name, "message": message})
    n += 1

# Write the conversation to a JSON file
with open('conversation.json', 'w') as f:
    json.dump(conversation, f, indent=4)

(Moderator): Discuss the following topic: 

"To what extent should hunting of wild boars be allowed in Germany particularly considering population control, socio-economical impacts, and ethical considerations?" 

Each of you is a specialist in the area and your task is to effectively communicate your position to the other parties

Please, also, comment on the suggestions raised by other parties. Explain your position.







> Entering new AgentExecutor chain...
```json
{
    "action": "duckduckgo_search",
    "action_input": "impact of wild boar hunting on German agriculture"
}
```
Observation: Hunting is a method commonly used in several European countries to reduce crop damages by wild boar Sus scrofa. However, results are still controversial and poorly treated. Using data on official claims (i.e., damages to crops) and wild boar local counts and hunting bags collected from 2019 to 2022, the purpose of this work was to evaluate the effect of the hunting system (divided into ... Abst

In [ ]:
#@title Creating summary and suggestions based on the simulated discussion
import json
import openai

# Load your OpenAI API key
openai.api_key = api_key

# Load the conversation from the JSON file
file_path = '/content/drive/MyDrive/Multiagent_conversation_26062024/conversation.json'
with open(file_path, 'r') as file:
    conversation_data = json.load(file)

word_limit_summary=150
# Making a summary in the format from Abdelnahbi et al. 2024.
topic_specifier_prompt = [
    SystemMessage(content="Make a summary of the discussion"),
    HumanMessage(
        content=f"""
        Make a summary of the discussion : {conversation_data} and
        identify a clear and precise formulation of the

        (i) topic related issues and
        (ii) issue specific action options

        (i) The topic related issue formulation should take into the account the following:
          The issues should cover different aspects of the problem
          The issue definition should be agreed between stakeholders

        (ii) The issue specific action options should take into the account the following:
          The action options should reflect the different degrees of one specific action.

          For example if an action is to impose the control over the certain activity, the
          action specific options can be: "strict", "mild" and "no" control. Apply this logic
          for the other actions

          For each issue suggest 3 to 5 issue specific action options.

          The response should be in the following format, where for each Issue (e.g. Issue A) a list of subissues (e.g A1, A2 etc.) is provided.

          Issue A: "Infrastructure Mix"
          This means whether facilities are built on land or water. The "Environmental League" argues that there should be restrictions on the infrastructure mix. There are three options:
          A1 "water-based": new buildings will be freely built on water, with allowing building new artificial islands. This is the least restrictive option for SportCo.
          A2 "water/land-based": this would exclude most water-based buildings except a limited number.
          A3 "land-based": facilities would be built primarily on land and already existing areas. SportCo has less freedom in building new facilities.
          =================
          Issue B: "Ecological Impact"
          The "Environmental League" argues that this project might damage local dolphins and sea lion populations. There are also here three options:

          B1 "some damage": permanent damage but within federal guidelines.
          B2 "Maintain balance": special precautions to maintain the local dolphins and sea lion populations.
          B3 "Improve": include efforts to improve the environment.

          =================
          Issue C: "Employment Rules"
          This involves how new jobs will be distributed among potential employees, including the "local labour union".

          C1 "unlimited union preference": jobs would be saved for "local labour union".
          C2 "Union quota of 2:1": ratio of the "local labour union" to others would be 2:1.
          C3 "Union quota of 1:1": ratio of "local labour union" to others would be 1:1.
          C4 "No Union preference" no special quote to "local labour union".

          =================
          Issue D: "Federal Loan"
          This involves the fund paid by the "Department of Tourism" (represented by you) as a loan to SportCo. Options include:
          D1: $3 billion.
          D2: $2 billion.
          D3: $1 billion.
          D4: no federal loan.

          =================

          Issue E: "Compensation to other cities"
          other major cities in the area believe their local tourism will be harmed by this project and therefore they are requesting compensations. Options include

          E1: SportCo pays $600 million to "other cities".
          E2: SportCo pays $450 million to "other cities".
          E3: SportCo pays $300 million to "other cities".
          E4: SportCo pays $150 million to "other cities".
          E5: SportCo pays no compensation to "other cities"



        """
    ),
]
summary = ChatOpenAI(temperature=1.0,openai_api_key = api_key)(topic_specifier_prompt).content

#word_limit_suggestion=150
#topic_specifier_prompt = [
#    SystemMessage(content="You are a government representative"),
#    HumanMessage(
#        content=f"""
#        Based on {summary} can you suggest what can be concrete steps the government can implement?
#        Government regulates the hunting quota for the species, as well as the processes of the license issue.
#        Limit your summary response to {word_limit_suggestion}
#        """
#    ),
#]
#suggestion = ChatOpenAI(temperature=1.0,openai_api_key = api_key)(topic_specifier_prompt).content

In [ ]:
#@title Visualizing the chat
import json
from IPython.display import HTML, display

# Function to read and process the conversation from a JSON file
def read_conversation(file_path):
    with open(file_path, 'r') as file:
        return json.load(file)

# Generate a dynamic color mapping for participants
def generate_color_mapping(conversation, base_colors):
    unique_names = {entry['name'] for entry in conversation}
    color_palette = ["#E2F7CB", "#D9FDD3", "#BFE1B0", "#A5D6A7", "#81C784"]  # Shades similar to WhatsApp
    colors = {name: color_palette[i % len(color_palette)] for i, name in enumerate(unique_names)}
    colors.update(base_colors)  # Update with base colors to ensure Moderator has a specific color
    return colors

# Base colors for specific roles
base_colors = {
    "Moderator": "lightgrey",  # WhatsApp's light grey for Moderator
}
summary_color = "#FFCDD2"  # Light red for summary
suggestion_color = "#CFD8DC"  # Light grey for suggestion

# Read conversation from the provided file
file_path = '/content/drive/MyDrive/Multiagent_conversation_26062024/conversation.json'
conversation = read_conversation(file_path)

# Generate dynamic colors for participants
colors = generate_color_mapping(conversation, base_colors)

# Display the conversation
for entry in conversation:
    name = entry.get("name")
    message = entry.get("message")
    color = colors.get(name, "lightgrey")  # Fallback color
    display(HTML(f"<div style='background-color:{color};padding:10px;margin:10px;border-radius:5px;'>\
                    <b>{name}:</b> {message}</div>"))

# Assume summary and suggestion are defined somewhere in your script

# Display the summary and suggestions
display(HTML(f"<div style='background-color:{summary_color};padding:10px;margin:10px;border-radius:5px;'>\
                <b>Summary: </b> {summary}</div>"))
#display(HTML(f"<div style='background-color:{suggestion_color};padding:10px;margin:10px;border-radius:5px;'>\
#                <b>Suggestion: </b> {suggestion}</div>"))
